"""Property Market Analysis – Data Integration & Feature Engineering

This notebook:
1. Loads and merges 4 Property Finder CSV files.
2. Loads the already cleaned Aqarmap dataset (aqarmap_cleaned_properties.csv).
3. Merges both sources, standardises columns, cleans missing values & duplicates.
4. Engineers features: price_per_sqm, estimated ROI, amenities flags, district.
5. Exports a single CSV ready for dashboard visualisations (heatmap, bar chart, etc.).

"""

## Import libararies

In [2]:
import pandas as pd
import numpy as np
import re
import glob


##Load & Merge the 4 Property Finder Files

In [5]:
print("="*60)
print("STEP 1: Loading Property Finder files")
print("="*60)

# Look for all CSV files whose names start with 'property_finder_'
pf_files = [
    "/content/big_data_project.listings.csv",
    "/content/prop_finder_listings_nasr-city.csv",
    "/content/propertyfinder.properties-1.csv",
    "/content/propertyfinder.properties.csv"
]

if not pf_files:
    raise FileNotFoundError("No Property Finder files found! "
                            "Please upload them as property_finder_1.csv, ...")

print(f"Found {len(pf_files)} files: {pf_files}")

pf_dfs = []
for f in pf_files:
    df_temp = pd.read_csv(f, low_memory=False)
    df_temp['source'] = 'property_finder'   # Add source column
    pf_dfs.append(df_temp)

df_pf = pd.concat(pf_dfs, ignore_index=True)
print(f"Combined Property Finder shape: {df_pf.shape}")

STEP 1: Loading Property Finder files
Found 4 files: ['/content/big_data_project.listings.csv', '/content/prop_finder_listings_nasr-city.csv', '/content/propertyfinder.properties-1.csv', '/content/propertyfinder.properties.csv']
Combined Property Finder shape: (27192, 37)


##Load Aqarmap Cleaned Data

In [6]:
print("\n" + "="*60)
print("STEP 2: Loading Aqarmap cleaned data")
print("="*60)

aqarmap_file = "/content/aqarmap_cleaned_properties.csv"
df_aq = pd.read_csv(aqarmap_file, low_memory=False)
df_aq['source'] = 'aqarmap'
print(f"Aqarmap shape: {df_aq.shape}")



STEP 2: Loading Aqarmap cleaned data
Aqarmap shape: (18245, 27)


##Standardise Column Names for Merging

In [12]:
print("\n" + "="*60)
print("STEP 3: Standardising column names")
print("="*60)

# --- For Property Finder: map possible column names to unified names ---
pf_col_map = {
    'price': 'unified_price',
    'Price': 'unified_price',
    'area': 'unified_area',
    'Area': 'unified_area',
    'sqm': 'unified_area',
    'rooms': 'unified_rooms',
    'bedrooms': 'unified_rooms',
    'bathrooms': 'unified_bathrooms',
    'location': 'unified_location',
    'address': 'unified_location',
    'property_type': 'unified_property_type',
    'type': 'unified_property_type',
    'amenities': 'unified_amenities',
    # New mappings for additional columns
    'city': 'city',
    'compound': 'compound',
    'is_furnished': 'finished', # Assuming 'is_furnished' from PF maps to 'finished'
    'updated_at': 'listing_date' # Assuming 'updated_at' from PF maps to 'listing_date'
}
# Apply only columns that exist
df_pf.rename(columns={k: v for k, v in pf_col_map.items() if k in df_pf.columns}, inplace=True)

# --- For Aqarmap (already cleaned by cleaner.py, but ensure unified names) ---
aq_col_map = {
    'price': 'unified_price',
    'area': 'unified_area',
    'rooms': 'unified_rooms',
    'bathrooms': 'unified_bathrooms',
    'location': 'unified_location',
    'property_type': 'unified_property_type',
    'amenities': 'unified_amenities',
    # New mappings for additional columns (Aqarmap columns are generally already well-named)
    'governorate': 'governorate',
    'city': 'city',
    'compound': 'compound',
    'developer': 'developer',
    'finished': 'finished',
    'level': 'level',
    'listing_date': 'listing_date'
}
df_aq.rename(columns={k: v for k, v in aq_col_map.items() if k in df_aq.columns}, inplace=True)

# Ensure DataFrames have unique column names after renaming, keeping the first occurrence
df_pf = df_pf.loc[:, ~df_pf.columns.duplicated()]
df_aq = df_aq.loc[:, ~df_aq.columns.duplicated()]

# Ensure both DataFrames have the same core columns
required_cols = [
    'unified_price', 'unified_area', 'unified_rooms',
    'unified_bathrooms', 'unified_location', 'unified_property_type',
    'unified_amenities', 'source', 'url',
    'governorate', 'city', 'compound', 'developer', 'finished', 'level', 'listing_date'
]
for col in required_cols:
    if col not in df_pf:
        df_pf[col] = np.nan
    if col not in df_aq:
        df_aq[col] = np.nan


STEP 3: Standardising column names


/tmp/ipykernel_33020/2298708210.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pf.rename(columns={k: v for k, v in pf_col_map.items() if k in df_pf.columns}, inplace=True)


##Merge Both Sources

In [14]:
print("\n" + "="*60)
print("STEP 4: Merging Property Finder & Aqarmap")
print("="*60)

# Ensure 'url' column exists in both dataframes. It's important for deduplication later.
# If 'url' is missing from either, add it and fill with NaN.
if 'url' not in df_pf.columns:
    df_pf['url'] = np.nan
if 'url' not in df_aq.columns:
    df_aq['url'] = np.nan

# Define the full set of columns that we want to keep in the merged dataframe.
# This includes the previously defined 'required_cols' and 'url'.
# Using 'set' ensures uniqueness, then converting to list preserves order implicitly but is not strictly necessary for concat.
final_subset_cols = list(set(required_cols + ['url']))

# Filter both dataframes to only include these essential columns.
# This proactively removes any other columns that might cause conflicts or are not needed.
df_pf_filtered = df_pf[final_subset_cols]
df_aq_filtered = df_aq[final_subset_cols]

df_merged = pd.concat([df_pf_filtered, df_aq_filtered], ignore_index=True, sort=False)
print(f"Merged shape before cleaning: {df_merged.shape}")


STEP 4: Merging Property Finder & Aqarmap
Merged shape before cleaning: (45437, 16)


##Data Cleaning

In [20]:
DATA_PATH = '/content/'
print("\n" + "="*60)
print("STEP 5: Data cleaning")
print("="*60)

# ---- 6.1 Convert numeric fields (extract numbers from strings) ----
def extract_numbers(text):
    if pd.isna(text):
        return np.nan
    # Remove decimal points and trailing zeros, then keep only digits
    cleaned = str(text).split('.')[0].replace(',', '')
    numbers = re.findall(r'\d+', cleaned)
    return int(''.join(numbers)) if numbers else np.nan

for col in ['unified_price', 'unified_area', 'unified_rooms', 'unified_bathrooms']:
    if col in df_merged.columns:
        df_merged[col] = df_merged[col].apply(extract_numbers)

# ---- 6.2 Remove rows missing essential ML features (price and area) ----
df_merged = df_merged.dropna(subset=['unified_price', 'unified_area'])

# ---- 6.3 Remove duplicates (based on URL if exists, else on price+area+location) ----
if 'url' in df_merged.columns:
    df_merged = df_merged.drop_duplicates(subset=['url'], keep='last')
else:
    # Fallback: use a combination of price, area, and location
    df_merged = df_merged.drop_duplicates(subset=['unified_price', 'unified_area', 'unified_location'],
                                          keep='last')

# ---- 6.4 Outlier removal (sanity bounds for Egyptian market) ----
# Remove unrealistic prices (< 10,000 EGP or > 200,000,000 EGP) and area (< 10 m² or > 2000 m²)
df_merged = df_merged[(df_merged['unified_price'] >= 10000) & (df_merged['unified_price'] <= 200_000_000)]
df_merged = df_merged[(df_merged['unified_area'] >= 10) & (df_merged['unified_area'] <= 2000)]

# ---- 6.5 Fill missing categoricals with 'unknown' ----
cat_cols = ['unified_property_type', 'unified_location']
for col in cat_cols:
    if col in df_merged.columns:
        df_merged[col] = df_merged[col].fillna('unknown')

# Fill missing bathrooms (assume 1) and rooms (assume median by area)
if 'unified_bathrooms' in df_merged.columns:
    df_merged['unified_bathrooms'] = df_merged['unified_bathrooms'].fillna(1)
if 'unified_rooms' in df_merged.columns:
    df_merged['unified_rooms'] = df_merged['unified_rooms'].fillna(df_merged.groupby('unified_area')['unified_rooms'].transform('median'))
    df_merged['unified_rooms'] = df_merged['unified_rooms'].fillna(2)  # fallback

print(f"Shape after cleaning: {df_merged.shape}")


STEP 5: Data cleaning
Shape after cleaning: (25165, 29)


In [22]:
print("\n" + "="*60)
print("STEP 5: Data cleaning")
print("="*60)

# ---- 6.1 Convert numeric fields (extract numbers from strings) ----
def extract_numbers(text):
    if pd.isna(text):
        return np.nan
    # Remove decimal points and trailing zeros, then keep only digits
    cleaned = str(text).split('.')[0].replace(',', '')
    numbers = re.findall(r'\d+', cleaned)
    return int(''.join(numbers)) if numbers else np.nan

for col in ['unified_price', 'unified_area', 'unified_rooms', 'unified_bathrooms']:
    if col in df_merged.columns:
        df_merged[col] = df_merged[col].apply(extract_numbers)

# ---- 6.2 Remove rows missing essential ML features (price and area) ----
df_merged = df_merged.dropna(subset=['unified_price', 'unified_area'])

# ---- 6.3 Remove duplicates (based on URL if exists, else on price+area+location) ----
if 'url' in df_merged.columns:
    df_merged = df_merged.drop_duplicates(subset=['url'], keep='last')
else:
    # Fallback: use a combination of price, area, and location
    df_merged = df_merged.drop_duplicates(subset=['unified_price', 'unified_area', 'unified_location'],
                                          keep='last')

# ---- 6.4 Outlier removal (sanity bounds for Egyptian market) ----
# Remove unrealistic prices (< 10,000 EGP or > 200,000,000 EGP) and area (< 10 m² or > 2000 m²)
df_merged = df_merged[(df_merged['unified_price'] >= 10000) & (df_merged['unified_price'] <= 200_000_000)]
df_merged = df_merged[(df_merged['unified_area'] >= 10) & (df_merged['unified_area'] <= 2000)]

# ---- 6.5 Fill missing categoricals with 'unknown' ----
cat_cols = ['unified_property_type', 'unified_location']
for col in cat_cols:
    if col in df_merged.columns:
        df_merged[col] = df_merged[col].fillna('unknown')

# Fill missing bathrooms (assume 1) and rooms (assume median by area)
if 'unified_bathrooms' in df_merged.columns:
    df_merged['unified_bathrooms'] = df_merged['unified_bathrooms'].fillna(1)
if 'unified_rooms' in df_merged.columns:
    df_merged['unified_rooms'] = df_merged['unified_rooms'].fillna(df_merged.groupby('unified_area')['unified_rooms'].transform('median'))
    df_merged['unified_rooms'] = df_merged['unified_rooms'].fillna(2)  # fallback

print(f"Shape after cleaning: {df_merged.shape}")


STEP 5: Data cleaning
Shape after cleaning: (25165, 29)


##Feature Engineering

In [23]:
print("\n" + "="*60)
print("STEP 6: Feature engineering")
print("="*60)

# ---- 7.1 Price per square metre ----
df_merged['price_per_sqm'] = df_merged['unified_price'] / df_merged['unified_area']

# ---- 7.2 Amenities flags (same as in cleaner.py) ----
if 'unified_amenities' in df_merged.columns:
    df_merged['unified_amenities'] = df_merged['unified_amenities'].fillna('')
    key_amenities = ['elevator', 'security', 'balcony', 'pool', 'garden', 'parking']
    for amenity in key_amenities:
        df_merged[f'has_{amenity}'] = df_merged['unified_amenities'].str.lower().str.contains(amenity).astype(int)

# ---- 7.3 District extraction (for heatmap) ----
# We build a list of known Egyptian districts (Cairo, Alexandria, Giza, etc.)
# The function extracts the first matching district from the location string.
def extract_district(location_str):
    if pd.isna(location_str):
        return 'unknown'
    l = str(location_str).lower()
    districts = {
        'new cairo': 'New Cairo',
        '5th settlement': 'New Cairo',
        'maadi': 'Maadi',
        'heliopolis': 'Heliopolis',
        'nasr city': 'Nasr City',
        'sheikh zayed': 'Sheikh Zayed',
        'october 6': '6th October',
        'rehab': 'Rehab City',
        'madinty': 'Madinaty',
        'alexandria': 'Alexandria',
        'smouha': 'Smouha',
        'downtown': 'Downtown Cairo',
        'garden city': 'Garden City',
        'zamalek': 'Zamalek',
        'mokattam': 'Mokattam'
    }
    for key, val in districts.items():
        if key in l:
            return val
    return 'Other'

df_merged['district'] = df_merged['unified_location'].apply(extract_district)

# ---- 7.4 Estimate ROI (based on district-level gross rental yield) ----
# Typical annual gross rental yields for Egyptian districts (Example values)
yield_table = {
    'New Cairo': 7.5,
    'Maadi': 8.0,
    'Heliopolis': 7.0,
    'Nasr City': 8.5,
    'Sheikh Zayed': 7.2,
    '6th October': 8.0,
    'Rehab City': 7.8,
    'Madinaty': 7.5,
    'Alexandria': 6.5,
    'Smouha': 7.0,
    'Downtown Cairo': 5.5,
    'Garden City': 5.0,
    'Zamalek': 5.5,
    'Mokattam': 6.0,
    'Other': 6.5,
    'unknown': 6.5
}

df_merged['gross_rental_yield_pct'] = df_merged['district'].map(yield_table).fillna(6.5)
df_merged['estimated_annual_rent'] = df_merged['unified_price'] * (df_merged['gross_rental_yield_pct'] / 100)
df_merged['estimated_roi_percent'] = df_merged['gross_rental_yield_pct']   # ROI = yield for buy-to-let

# (Optional) Monthly rent estimate
df_merged['estimated_monthly_rent'] = df_merged['estimated_annual_rent'] / 12


STEP 6: Feature engineering


##Additional Metrics (for dashboard)

In [24]:

# Median price per district (can be used for comparison)
df_merged['median_price_per_district'] = df_merged.groupby('district')['unified_price'].transform('median')


##Save Final Cleaned & Enriched Dataset

In [21]:
print("\n" + "="*60)
print("STEP 7: Saving final CSV")
print("="*60)

final_cols = [
    'source', 'unified_price', 'unified_area', 'price_per_sqm',
    'unified_rooms', 'unified_bathrooms', 'unified_property_type',
    'district', 'unified_location', 'gross_rental_yield_pct',
    'estimated_annual_rent', 'estimated_roi_percent', 'estimated_monthly_rent',
    'has_elevator', 'has_security', 'has_balcony', 'has_pool', 'has_garden', 'has_parking',
    'median_price_per_district'
]
# Keep only columns that actually exist
final_cols = [c for c in final_cols if c in df_merged.columns]

df_final = df_merged[final_cols].copy()
output_file = DATA_PATH + "final_unified_property_data.csv"
df_final.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"✅ Final data saved to: {output_file}")
print(f"Final shape: {df_final.shape}")
print("\nFirst 5 rows:")
print(df_final.head())


STEP 7: Saving final CSV
✅ Final data saved to: /content/final_unified_property_data.csv
Final shape: (25165, 20)

First 5 rows:
            source  unified_price  unified_area  price_per_sqm  unified_rooms  \
0  property_finder       10850000           164   66158.536585              3   
1  property_finder       15272500           176   86775.568182              3   
2  property_finder       36000000           650   55384.615385              5   
3  property_finder        3935000           158   24905.063291              3   
4  property_finder        3530000            72   49027.777778              0   

   unified_bathrooms unified_property_type district unified_location  \
0                  2               unknown    Other          unknown   
1                  3               unknown    Other          unknown   
2                  6               unknown    Other          unknown   
3                  3               unknown    Other          unknown   
4                  1   

##Schema of the Final CSV

In [25]:
print("\n" + "="*60)
print("FINAL CSV SCHEMA")
print("="*60)

schema_info = []
for col in df_final.columns:
    dtype = df_final[col].dtype
    non_null = df_final[col].notna().sum()
    sample = str(df_final[col].iloc[0]) if len(df_final) > 0 else 'N/A'
    schema_info.append([col, dtype, non_null, sample])

schema_df = pd.DataFrame(schema_info, columns=['Column Name', 'Data Type', 'Non\u2011Null Count', 'Example Value'])
print(schema_df.to_string(index=False))


FINAL CSV SCHEMA
              Column Name Data Type  Non‑Null Count      Example Value
                   source    object           25165    property_finder
            unified_price     int64           25165           10850000
             unified_area     int64           25165                164
            price_per_sqm   float64           25165  66158.53658536586
            unified_rooms     int64           25165                  3
        unified_bathrooms     int64           25165                  2
    unified_property_type    object           25165            unknown
                 district    object           25165              Other
         unified_location    object           25165            unknown
   gross_rental_yield_pct   float64           25165                6.5
    estimated_annual_rent   float64           25165           705250.0
    estimated_roi_percent   float64           25165                6.5
   estimated_monthly_rent   float64           25165 58770.8